In [108]:
import os

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]='0'

In [109]:
import torch
import random
import numpy as np

def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
set_seeds(42)

In [110]:
import matplotlib.pyplot as plt
def show_images(images, scores, test_artist, train_artists):
    n: int = len(images)
    f = plt.figure(figsize=(16, 2))
    for i in range(n):
        # Debug, plot figure
        ax = f.add_subplot(1, n, i + 1)
        if i==0:
            pass
            ax.title.set_text(test_artist)
        else:
            ax.title.set_text(str(np.round(scores[i-1], 4))+'\n'+train_artists[i-1])
            ax.axis('off')
        if images[i]==None:
            pass
        else:
            plt.imshow(images[i])

    plt.show(block=True)

In [111]:
from datasets import load_dataset

In [112]:
import pickle

In [ ]:
with open('../../data/indices/5000-0.5/idx-train.pkl', 'rb')  as handle:
    idx_train = pickle.load(handle)
len(idx_train)   

In [114]:
# with open('../../data/indices/5000-0.5/idx-val.pkl', 'rb')  as handle:
#     idx_val = pickle.load(handle)
# len(idx_val)

In [115]:
import pandas as pd

df = pd.read_csv('/home/yourname/.cache/kagglehub/datasets/alexanderliao/artbench10/versions/2/ArtBench-10.csv')
#df = pd.read_csv('../../../../codes/artbench/ArtBench-10.csv')
df.head()

,name,artist,url,is_public_domain,length,width,label,split,cifar_index
0,frank-omeara_towards-night-and-winter.jpg,frank-omeara,https://uploads5.wikiart.org/00316/images/fran...,True,800,657,impressionism,train,43186
1,goldstein-grigoriy_morning.jpg,goldstein-grigoriy,https://uploads5.wikiart.org/images/grigoriy-g...,True,521,499,impressionism,train,41151
2,georges-lemmen_man-reading.jpg,georges-lemmen,https://uploads6.wikiart.org/images/georges-le...,True,800,612,impressionism,train,9754
3,theodor-aman_port-of-constantza-1882.jpg,theodor-aman,https://uploads6.wikiart.org/images/theodor-am...,True,560,336,impressionism,train,44244
4,niccolo-cannicci_il-passo-della-futa-1914.jpg,niccolo-cannicci,https://uploads3.wikiart.org/images/niccolo-ca...,True,2400,2322,impressionism,train,46885


In [116]:
df['path'] = df.apply(lambda x: "/home/yourname/neurips/artbench-merged/{}/{}".format(x['label'], x['name']), axis=1)
#df['path'] = df.apply(lambda x: "../../../../codes/artbench/data/artbench-10-imagefolder/{}/{}".format(x['label'], x['name']), axis=1)
df.head()

,name,artist,url,is_public_domain,length,width,label,split,cifar_index,path
0,frank-omeara_towards-night-and-winter.jpg,frank-omeara,https://uploads5.wikiart.org/00316/images/fran...,True,800,657,impressionism,train,43186,/home/yourname/neurips/artbench-merged/impre...
1,goldstein-grigoriy_morning.jpg,goldstein-grigoriy,https://uploads5.wikiart.org/images/grigoriy-g...,True,521,499,impressionism,train,41151,/home/yourname/neurips/artbench-merged/impre...
2,georges-lemmen_man-reading.jpg,georges-lemmen,https://uploads6.wikiart.org/images/georges-le...,True,800,612,impressionism,train,9754,/home/yourname/neurips/artbench-merged/impre...
3,theodor-aman_port-of-constantza-1882.jpg,theodor-aman,https://uploads6.wikiart.org/images/theodor-am...,True,560,336,impressionism,train,44244,/home/yourname/neurips/artbench-merged/impre...
4,niccolo-cannicci_il-passo-della-futa-1914.jpg,niccolo-cannicci,https://uploads3.wikiart.org/images/niccolo-ca...,True,2400,2322,impressionism,train,46885,/home/yourname/neurips/artbench-merged/impre...


In [ ]:
from datasets import Dataset, load_dataset, Image

train_dataset = Dataset.from_dict({"image": df.loc[idx_train]['path'].tolist(),
                                   "label": df.loc[idx_train]['label'].tolist(),
                                  }).cast_column("image", Image())
train_dataset[0]["image"]

In [ ]:
import pandas as pd
df = pd.DataFrame()
df['label'] = ['ukiyo_e']*500+['post_impressionism']*500
df['path'] = ['{}/{}.png'.format('../../saved/5000-0.5/gen', i) for i in range(1000)]

from datasets import DatasetDict, Dataset, load_dataset, Image
dataset = DatasetDict({
"train": Dataset.from_dict({
    "image": df['path'].tolist(),
    "label": df['label'].tolist(),
}).cast_column("image", Image()),})
val_dataset = dataset["train"]
val_dataset[0]["image"]

In [ ]:
import numpy as np
import torch
from pkg_resources import packaging

print("Torch version:", torch.__version__)

In [ ]:
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import sys
# 1. Uninstall the current clip
!{sys.executable} -m pip uninstall -y clip
# 2. Install the correct one from the official source
!{sys.executable} -m pip install git+https://github.com/openai/CLIP.git
# 3. Re-install ftfy and regex which are CLIP dependencies
!{sys.executable} -m pip install ftfy regex

In [120]:
import clip
print(clip.__file__)

/home/yourname/.conda/envs/dtrak2/lib/python3.10/site-packages/clip/__init__.py


In [ ]:
import clip

clip.available_models()

In [ ]:
model, preprocess = clip.load("ViT-L/14@336px")
#model, preprocess = clip.load("ViT-B/32")
#model, preprocess = clip.load("RN50x64")
model.cuda().eval()
input_resolution = model.visual.input_resolution
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Input resolution:", input_resolution)
print("Context length:", context_length)
print("Vocab size:", vocab_size)

In [ ]:
preprocess

In [ ]:
train_features = []
for i in range(0, len(train_dataset), 32):
    batch = train_dataset[i:i+32]['image']
    batch = [preprocess(b) for b in batch]
    batch = torch.tensor(np.stack(batch)).cuda()
    with torch.no_grad():
        image_features = model.encode_image(batch).float()
    print(image_features.size())
    train_features.append(image_features.cpu().numpy())

In [160]:
train_features_array = np.vstack(train_features)
train_features_array.shape

(5000, 768)

In [ ]:
val_features = []
for i in range(0, len(val_dataset), 32):
    batch = val_dataset[i:i+32]['image']
    batch = [preprocess(b) for b in batch]
    batch = torch.tensor(np.stack(batch)).cuda()
    with torch.no_grad():
        image_features = model.encode_image(batch).float()
    print(image_features.size())
    val_features.append(image_features.cpu().numpy())

In [ ]:
val_features_array = np.vstack(val_features)
val_features_array.shape

Here we pass the embeddings from the trained MLPs

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Redefine the exact same architecture
class ProjectionHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        x = self.mlp(x)
        return F.normalize(x, p=2, dim=1)

class MultiModalAlignmentModel(nn.Module):
    def __init__(self, img_in_dim=768, graph_in_dim=512, shared_dim=256):
        super().__init__()
        self.image_proj = ProjectionHead(input_dim=img_in_dim, hidden_dim=512, output_dim=shared_dim)
        self.graph_proj = ProjectionHead(input_dim=graph_in_dim, hidden_dim=512, output_dim=shared_dim)

    def forward(self, img_embeds, graph_embeds):
        return self.image_proj(img_embeds), self.graph_proj(graph_embeds)

# 2. Instantiate and Load the Weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiModalAlignmentModel(img_in_dim=768, graph_in_dim=512, shared_dim=256).to(device)

# Load the saved .pth file (make sure the path is correct)
#model.load_state_dict(torch.load('/home/yourname/contrastive_learning/multimodal_projection_heads.pth', map_location=device))
model.load_state_dict(torch.load('/home/yourname/contrastive_learning/projection_heads_v1/multimodal_projection_heads_vit336_node2vec.pth', map_location=device))


# CRITICAL: Put the model in evaluation mode
model.eval()
print("Trained MLPs successfully loaded!")

In [ ]:
print(f"Old train shape: {train_features_array.shape}")
print(f"Old val shape: {val_features_array.shape}")

with torch.no_grad(): # We are just evaluating, not training
    # 1. Convert numpy arrays to tensors and move to GPU/CPU
    train_tensor = torch.tensor(train_features_array, dtype=torch.float32).to(device)
    val_tensor = torch.tensor(val_features_array, dtype=torch.float32).to(device)
    
    # 2. Pass them ONLY through the image projection head
    shared_train_features = model.image_proj(train_tensor)
    shared_val_features = model.image_proj(val_tensor)
    
    # 3. Move back to CPU and convert back to numpy arrays
    shared_train_array = shared_train_features.cpu().numpy()
    shared_val_array = shared_val_features.cpu().numpy()

print(f"New shared train shape: {shared_train_array.shape}")
print(f"New shared val shape: {shared_val_array.shape}")

here we compute density (p(xj))

In [ ]:
# cell
import torch
import numpy as np

# 1. Move the shared features to GPU for fast matrix multiplication
train_shared_t = torch.tensor(shared_train_array).cuda()

# 2. Compute Self-Similarity Matrix (5000 x 5000)
# Because ProjectionHead already normalized the vectors, 
# mm() here is exactly the Cosine Similarity.
self_sim_matrix = torch.mm(train_shared_t, train_shared_t.t())

# 3. Calculate Density
# We sum the similarities for each image. 
# High sum = many similar neighbors = high redundancy.
# We subtract 1 to ignore the similarity of the image with itself (the diagonal).
density = self_sim_matrix.sum(dim=1) - 1

# 4. Convert Density to a Prior Score P(Xj)
# We use your formula: P(Xj) ∝ exp(-density)
# First, we normalize density to [0, 1] to prevent the exp from exploding/vanishing
density_min = density.min()
density_max = density.max()
norm_density = (density - density_min) / (density_max - density_min)

# Calculate the Prior: Unique images get higher weights
prior_xj = torch.exp(-norm_density)

# Convert to numpy for the final scoring step
prior_xj_np = prior_xj.cpu().numpy()

print(f"Prior calculated from {shared_train_array.shape[0]} training images.")
print(f"Prior range: [{prior_xj_np.min():.4f}, {prior_xj_np.max():.4f}]")

In [ ]:
# cell
# We use your standard visual similarity as the Likelihood P(C | Xj)
# Use your val_features_array and train_features_array (the high-dim ones)
val_norm = val_features_array / np.linalg.norm(val_features_array, axis=1, keepdims=True)
train_norm = train_features_array / np.linalg.norm(train_features_array, axis=1, keepdims=True)

# Likelihood: 1000 x 5000 matrix
likelihood = val_norm.dot(train_norm.T)

# 1. Standard Bayesian Score: P(C | Xj) * P(Xj)
# We multiply each column (training image) by its corresponding prior weight
bayesian_scores = likelihood * prior_xj_np

# 2. Bayesian Power-5 Score (usually performs best for LDS)
# We apply the power to the likelihood first, then weight by the prior
bayesian_power_5 = np.power(np.maximum(likelihood, 0), 5) * prior_xj_np

# Append to your list for the LDS evaluation
scores_list = []
scores_list.append(bayesian_scores)
scores_list.append(bayesian_power_5)

print("Bayesian scores computed successfully.")

In [167]:
with open('./gen_clip.pkl', 'wb') as handle:
    pickle.dump(scores_list, handle)

In [168]:
my_list = [
    0,1,2,3,
    4,5,6,7,
    8,9,10,11,
    12,13,14,15,
    16,17,18,19,
    20,21,22,23,
    24,25,26,27,
    28,29,30,31,
    32,33,34,35,
    36,37,38,39,
    40,41,42,43,
    44,45,46,47,
    48,49,50,51,
    52,53,54,55,
    56,57,58,59,
    60,61,62,63,
          ]

In [ ]:
loss_array_list = []

for i in my_list:
    for seed in [
        0,
                 1,
                 2,
                 # 3,
                 # 4,
                ]:
        for e_seed in [
            0, 
                       1, 
                       2
                      ]:
            with open('../../saved/5000-0.5/lds-val/sd-lora-sub-{}-{}/e-{}-gen.pkl'.format(i, seed, e_seed), 'rb')  as handle:
                loss_list = pickle.load(handle)
            margins = np.concatenate(loss_list, axis=-1) # -logp
            ####
            if (seed==0) and (e_seed)==0:
                loss_array = margins
            else:
                loss_array += margins
            
    loss_array = loss_array/(3*3)
    
    loss_array_list.append(loss_array)
lds_loss_array = np.stack(loss_array_list)
lds_loss_array.shape

In [ ]:
mask_array_list = []

for i in my_list:
    # print(i)
    with open('../../data/indices/5000-0.5/lds-val/sub-idx-{}.pkl'.format(i), 'rb')  as handle:
        sub_idx_train = pickle.load(handle)
    # print(len(sub_idx_train))
    mask_array = np.in1d(idx_train, sub_idx_train)
        
    mask_array_list.append(mask_array)
    
lds_mask_array = np.stack(mask_array_list)
lds_mask_array.shape

In [ ]:
lds_testset_correctness = lds_loss_array.mean(axis=1)
lds_testset_correctness.shape

In [ ]:
for j in range(4):
    plt.plot(lds_testset_correctness[:, j], color="C{}".format(j))
    # break
# plt.ylim(0.15, 0.2)

In [ ]:
# compute lds
from scipy.stats import spearmanr, pearsonr
####
# k = 48
# margins = lds_testset_correctness[k:]
# infl_est_ = _masked_dot(lds_testset_correctness[:k], lds_mask_array[:k]) - _masked_dot(lds_testset_correctness[:k], ~lds_mask_array[:k])
# # infl_est_ = _masked_dot(lds_testset_correctness[:], lds_mask_array[:]) - _masked_dot(lds_testset_correctness[:], ~lds_mask_array[:])
# preds = lds_mask_array[k:] @ infl_est_.T

margins = lds_testset_correctness
np.random.seed(0)
infl_est_ = -np.random.rand(1000, 5000)
# infl_est_ = -tmp
preds = lds_mask_array @ infl_est_.T
####
rs = []
ps = []

for ind in range(1000):
    r, p = spearmanr(preds[:, ind], margins[:, ind])
    # r, p = pearsonr(preds[:, ind], margins[:, ind])
    rs.append(r)
    ps.append(p)
    
rs, ps = np.array(rs), np.array(ps)
print(f'Correlation: {rs.mean():.3f} (avg p value {ps.mean():.6f})')

In [ ]:
# compute lds
from scipy.stats import spearmanr, pearsonr
####
# k = 48
# margins = lds_testset_correctness[k:]
# infl_est_ = _masked_dot(lds_testset_correctness[:k], lds_mask_array[:k]) - _masked_dot(lds_testset_correctness[:k], ~lds_mask_array[:k])
# # infl_est_ = _masked_dot(lds_testset_correctness[:], lds_mask_array[:]) - _masked_dot(lds_testset_correctness[:], ~lds_mask_array[:])
# preds = lds_mask_array[k:] @ infl_est_.T

margins = lds_testset_correctness
np.random.seed(1)
infl_est_ = -np.random.rand(1000, 5000)
# infl_est_ = -tmp
preds = lds_mask_array @ infl_est_.T
####
rs = []
ps = []

for ind in range(1000):
    r, p = spearmanr(preds[:, ind], margins[:, ind])
    # r, p = pearsonr(preds[:, ind], margins[:, ind])
    rs.append(r)
    ps.append(p)
    
rs, ps = np.array(rs), np.array(ps)
print(f'Correlation: {rs.mean():.3f} (avg p value {ps.mean():.6f})')

In [176]:
(0.005 + -0.001 + -0.004)/3.0

0.0

In [ ]:
# compute lds
from scipy.stats import spearmanr, pearsonr
####
# k = 48
# margins = lds_testset_correctness[k:]
# infl_est_ = _masked_dot(lds_testset_correctness[:k], lds_mask_array[:k]) - _masked_dot(lds_testset_correctness[:k], ~lds_mask_array[:k])
# # infl_est_ = _masked_dot(lds_testset_correctness[:], lds_mask_array[:]) - _masked_dot(lds_testset_correctness[:], ~lds_mask_array[:])
# preds = lds_mask_array[k:] @ infl_est_.T

margins = lds_testset_correctness
infl_est_ = -scores_list[0]
# infl_est_ = -tmp
preds = lds_mask_array @ infl_est_.T
####
rs = []
ps = []

for ind in range(1000):
    r, p = spearmanr(preds[:, ind], margins[:, ind])
    # r, p = pearsonr(preds[:, ind], margins[:, ind])
    rs.append(r)
    ps.append(p)
    
rs, ps = np.array(rs), np.array(ps)
print(f'Correlation: {rs.mean():.3f} (avg p value {ps.mean():.6f})')

In [178]:
my_data = {
    'margins': margins[:, 0],
    'preds': preds[:, 0]
}

In [ ]:
import seaborn as sns
sns.jointplot(data=my_data, x="margins", y="preds", kind="reg")

In [ ]:
# compute lds
from scipy.stats import spearmanr, pearsonr
####
# k = 48
# margins = lds_testset_correctness[k:]
# infl_est_ = _masked_dot(lds_testset_correctness[:k], lds_mask_array[:k]) - _masked_dot(lds_testset_correctness[:k], ~lds_mask_array[:k])
# # infl_est_ = _masked_dot(lds_testset_correctness[:], lds_mask_array[:]) - _masked_dot(lds_testset_correctness[:], ~lds_mask_array[:])
# preds = lds_mask_array[k:] @ infl_est_.T

margins = lds_testset_correctness
infl_est_ = -scores_list[1]
# infl_est_ = -tmp
preds = lds_mask_array @ infl_est_.T
####
rs = []
ps = []

for ind in range(1000):
    r, p = spearmanr(preds[:, ind], margins[:, ind])
    # r, p = pearsonr(preds[:, ind], margins[:, ind])
    rs.append(r)
    ps.append(p)
    
rs, ps = np.array(rs), np.array(ps)
print(f'Correlation: {rs.mean():.3f} (avg p value {ps.mean():.6f})')

In [181]:
my_data = {
    'margins': margins[:, 0],
    'preds': preds[:, 0]
}

In [ ]:
import seaborn as sns
sns.jointplot(data=my_data, x="margins", y="preds", kind="reg")

In [183]:
scores = scores_list[1]

In [184]:
i = 0

In [ ]:
D = -scores[i]
D.shape

In [ ]:
plt.plot(sorted(D))
# plt.axhline(y=0, c='red')

In [187]:
topK = np.arange(5000)[D.argsort()[0:5]]
topK

array([1484, 1035, 3834, 2264,  923])

In [188]:
plot_images = []
plot_images.append(val_dataset[i]['image'])
for idx in topK:
    plot_images.append(train_dataset[int(idx)]['image'])

In [ ]:
val_artist = ''
val_artist

In [ ]:
train_artist = []
for k in topK:
    tmp_artist = ''
    train_artist.append(tmp_artist)
train_artist   

In [ ]:
# full
show_images(plot_images, D[D.argsort()[0:5]], val_artist, train_artist)

In [192]:
#done